# 01 — Data Exploration & Pipeline Validation

**Purpose:** First contact with the ingestion layer. Goals for this notebook:
1. Confirm the cache + nba_stats pipeline works end-to-end
2. Identify what columns each endpoint actually returns (discovery)
3. Surface the top players by available metrics for the 2024-25 and 2025-26 seasons
4. Pull the Lakers roster and Luka's career arc
5. Document data quality gaps before moving to feature engineering

**Do not** re-run all cells against the live API repeatedly — the first run caches everything to `data/cache/`.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path so src.data.* is importable without installing the package
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/suveerdhawan/Desktop/Codex/lakers-trade-engine


In [2]:
import logging

import pandas as pd

# Show info-level logs from our pipeline in the notebook output
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

from src.data import cache as C
from src.data import nba_stats as N
from src.data.config import CACHE_DIR, CURRENT_SEASON, SEASON_STRINGS

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Cache dir : {CACHE_DIR}")
print(f"Current season: {CURRENT_SEASON}")
print(f"Season range: {SEASON_STRINGS[0]} -- {SEASON_STRINGS[-1]}")

Cache dir : /Users/suveerdhawan/Desktop/Codex/lakers-trade-engine/data/cache
Current season: 2025-26
Season range: 2019-20 -- 2026-27


---
## 1. Base Stats — 2024-25 Season

Pull league-wide per-game stats. First call hits the API and writes to cache; subsequent calls are instant.

In [3]:
SEASON_PREV = "2024-25"
SEASON_CURR = "2025-26"

stats_2425 = N.get_player_stats(SEASON_PREV, per_mode="PerGame")
print(f"Rows: {len(stats_2425)}")
print(f"Columns ({len(stats_2425.columns)}): {list(stats_2425.columns)}")
stats_2425.head(3)

23:47:58  src.data.cache  INFO  cache hit: nba_player_stats_2024-25_PerGame


Rows: 569
Columns (67): ['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT']


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.000,26,14,12,0.538,18.700,3.100,7.300,0.421,1.300,3.900,0.327,1.700,2.400,0.683,0.800,2.500,3.300,1.200,0.600,0.500,0.200,0.500,1.700,1.500,9.100,-0.700,16.500,2,0,16.300,424,364,414,239,305,248,228,388,196,181,319,147,129,419,272,258,271,343,394,355,330,201,247,207,219,300,292,159,44,281,1
1,1631260,AJ Green,AJ,1610612749,MIL,25.000,73,44,29,0.603,22.700,2.500,5.800,0.429,2.100,5.000,0.427,0.300,0.400,0.815,0.200,2.100,2.400,1.500,0.500,0.500,0.100,0.000,2.200,0.700,7.400,3.100,13.800,0,0,14.600,86,62,188,164,220,310,298,353,77,112,37,470,486,180,497,326,374,291,408,352,467,520,136,394,278,83,342,281,44,318,1
2,1642358,AJ Johnson,AJ,1610612764,WAS,20.000,29,8,21,0.276,22.000,2.800,7.300,0.385,0.800,3.100,0.267,1.100,1.300,0.865,0.300,1.800,2.000,2.600,1.200,0.400,0.100,0.600,1.700,0.900,7.600,-5.200,14.300,0,0,14.100,412,445,297,473,232,275,227,472,298,243,425,239,266,87,483,372,407,148,198,402,449,120,239,321,273,522,332,281,44,328,2


In [4]:
stats_2526 = N.get_player_stats(SEASON_CURR, per_mode="PerGame")
print(f"2025-26 rows: {len(stats_2526)}")
stats_2526.head(3)

23:48:00  src.data.cache  INFO  cache miss -- fetching: nba_player_stats_2025-26_PerGame
23:48:01  src.data.cache  INFO  cached 582 rows to nba_player_stats_2025-26_PerGame.parquet


2025-26 rows: 582


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,25.000,24,12,12,0.500,9.400,1.400,3.300,0.436,0.800,1.900,0.422,0.600,0.800,0.778,0.400,1.400,1.800,0.300,0.300,0.500,0.200,0.200,1.100,0.600,4.200,-0.400,8.500,0,0,8.400,433,395,419,303,486,454,442,363,316,362,53,413,425,284,450,423,443,554,529,373,399,445,439,450,445,286,481,284,38,467,1
1,1631260,AJ Green,AJ,1610612749,MIL,26.000,78,31,47,0.397,29.100,3.400,7.900,0.424,3.000,7.100,0.419,0.700,0.800,0.855,0.400,2.400,2.700,1.900,1.000,0.500,0.100,0.200,2.300,1.000,10.400,-1.900,17.400,0,0,19.200,28,188,32,387,109,237,211,403,18,28,59,377,415,124,452,282,337,245,279,373,464,439,121,335,201,398,292,284,38,231,1
2,1642358,AJ Johnson,AJ,1610612742,DAL,21.000,48,9,39,0.188,9.400,1.200,3.700,0.324,0.300,1.200,0.211,0.700,0.800,0.850,0.300,0.900,1.100,1.000,0.600,0.200,0.100,0.500,0.600,0.600,3.300,-1.100,6.400,0,0,6.300,310,432,76,538,485,486,421,552,456,436,487,368,403,132,487,506,510,409,398,508,503,178,520,456,475,339,502,284,38,501,2


---
## 2. Advanced Stats — Column Discovery

The advanced endpoint returns nba.com's own metrics. Key question: which columns are available, and what's the nba.com equivalent of Basketball Reference's PER / WS / BPM?

In [5]:
adv_2425 = N.get_player_advanced(SEASON_PREV)
print(f"Advanced cols ({len(adv_2425.columns)}): {list(adv_2425.columns)}")
adv_2425.head(3)

23:48:01  src.data.cache  INFO  cache miss -- fetching: nba_player_advanced_2024-25
23:48:02  src.data.cache  INFO  cached 569 rows to nba_player_advanced_2024-25.parquet


Advanced cols (79): ['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'sp_work_OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'sp_work_DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'sp_work_NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'E_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'sp_work_PACE', 'PIE', 'POSS', 'FGM', 'FGA', 'FGM_PG', 'FGA_PG', 'FG_PCT', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'E_OFF_RATING_RANK', 'OFF_RATING_RANK', 'sp_work_OFF_RATING_RANK', 'E_DEF_RATING_RANK', 'DEF_RATING_RANK', 'sp_work_DEF_RATING_RANK', 'E_NET_RATING_RANK', 'NET_RATING_RANK', 'sp_work_NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'E_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'USG_PCT_RANK', 'E_USG_PCT_RANK', 'E_PA

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM,FGA,FGM_PG,FGA_PG,FG_PCT,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,E_OFF_RATING_RANK,OFF_RATING_RANK,sp_work_OFF_RATING_RANK,E_DEF_RATING_RANK,DEF_RATING_RANK,sp_work_DEF_RATING_RANK,E_NET_RATING_RANK,NET_RATING_RANK,sp_work_NET_RATING_RANK,AST_PCT_RANK,AST_TO_RANK,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,E_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,USG_PCT_RANK,E_USG_PCT_RANK,E_PACE_RANK,PACE_RANK,sp_work_PACE_RANK,PIE_RANK,FGM_RANK,FGA_RANK,FGM_PG_RANK,FGA_PG_RANK,FG_PCT_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.000,26,14,12,0.538,18.700,109.100,109.700,109.700,112.600,111.100,111.100,-3.500,-1.400,-1.400,0.093,2.070,11.700,0.036,0.128,0.080,5.700,5.700,0.508,0.542,0.189,0.193,103.300,103.710,86.430,103.710,0.082,1050,80,190,3.100,7.300,0.421,424,364,146,239,305,320,310,310,350,277,277,353,283,283,384,186,461,315,249,291,50,51,368,365,210,217,119,122,122,329,352,340,248,228,388,1
1,1631260,AJ Green,AJ,1610612749,MIL,25.000,73,44,29,0.603,22.700,113.500,114.000,114.000,106.900,107.600,107.600,6.600,6.400,6.400,0.086,2.700,18.600,0.011,0.087,0.051,6.900,6.900,0.612,0.621,0.123,0.127,101.390,100.840,84.030,100.840,0.058,3487,182,424,2.500,5.800,0.429,86,62,362,164,220,150,149,149,110,122,122,85,94,94,406,90,252,532,442,489,85,85,62,87,496,499,280,312,312,484,231,220,310,298,353,1
2,1642358,AJ Johnson,AJ,1610612764,WAS,20.000,29,8,21,0.276,22.000,103.500,104.100,104.100,114.800,115.600,115.600,-11.300,-11.500,-11.500,0.178,2.170,22.300,0.011,0.073,0.042,10.300,10.300,0.441,0.480,0.173,0.178,101.460,100.830,84.020,100.830,0.053,1342,82,213,2.800,7.300,0.385,412,445,253,473,232,478,462,462,448,453,453,492,497,497,168,154,147,529,515,544,330,335,493,490,272,276,271,313,313,503,348,325,275,227,472,2


In [6]:
adv_2526 = N.get_player_advanced(SEASON_CURR)
print(f"2025-26 advanced rows: {len(adv_2526)}")

23:48:02  src.data.cache  INFO  cache miss -- fetching: nba_player_advanced_2025-26
23:48:03  src.data.cache  INFO  cached 582 rows to nba_player_advanced_2025-26.parquet


2025-26 advanced rows: 582


---
## 3. Top 20 by PIE (nba.com's version of PER)

PIE (Player Impact Estimate) is nba.com's catch-all efficiency metric — roughly analogous to PER but based on box score contributions relative to league average. Higher = more impactful.

Note: True PER (Hollinger) and WS/48 (BBRef) are not available from nba_api. Phase 0.2 will add Basketball Reference ingestion.

In [7]:
if not adv_2425.empty and "PIE" in adv_2425.columns:
    min_gp = 30
    top_pie = (
        adv_2425[adv_2425["GP"] >= min_gp]
        .sort_values("PIE", ascending=False)
        .head(20)[["PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "GP", "MIN",
                    "PIE", "NET_RATING", "OFF_RATING", "DEF_RATING",
                    "TS_PCT", "USG_PCT"]]
        .reset_index(drop=True)
    )
    top_pie.index += 1
    print(f"Top 20 by PIE, 2024-25 (min {min_gp} GP)")
    display(top_pie)
else:
    print("PIE column not available -- columns returned:", list(adv_2425.columns))

Top 20 by PIE, 2024-25 (min 30 GP)


,PLAYER_NAME,TEAM_ABBREVIATION,AGE,GP,MIN,PIE,NET_RATING,OFF_RATING,DEF_RATING,TS_PCT,USG_PCT
1,Giannis Antetokounmpo,MIL,30.000,67,34.200,0.210,7.200,118.900,111.700,0.625,0.346
2,Nikola Jokić,DEN,30.000,70,36.700,0.206,10.500,125.600,115.100,0.663,0.285
3,Shai Gilgeous-Alexander,OKC,26.000,76,34.200,0.199,16.700,122.400,105.700,0.637,0.336
4,Anthony Davis,DAL,32.000,51,33.400,0.179,0.500,113.300,112.900,0.588,0.301
5,Luka Dončić,LAL,26.000,50,35.400,0.170,9.100,119.600,110.500,0.587,0.328
6,LeBron James,LAL,40.000,70,34.900,0.169,-1.300,112.700,114.000,0.604,0.291
7,Victor Wembanyama,SAS,21.000,46,33.200,0.168,2.600,112.500,110.000,0.594,0.300
8,Zion Williamson,NOP,24.000,30,28.600,0.167,-1.800,113.700,115.500,0.600,0.325
9,Moritz Wagner,ORL,28.000,30,18.800,0.165,-0.900,106.100,107.000,0.649,0.254
10,Domantas Sabonis,SAC,29.000,70,34.700,0.164,2.600,116.500,113.900,0.655,0.213


---
## 4. Top 20 by Net Rating (min 30 GP, min 20 min/g)

Net Rating (points scored minus points allowed per 100 possessions) is the cleanest single-number impact metric available from the nba_api advanced endpoint. It proxies WS/48 reasonably well for high-usage players.

In [8]:
if not adv_2425.empty and "NET_RATING" in adv_2425.columns:
    top_net = (
        adv_2425[(adv_2425["GP"] >= 30) & (adv_2425["MIN"] >= 20)]
        .sort_values("NET_RATING", ascending=False)
        .head(20)[["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN",
                    "NET_RATING", "OFF_RATING", "DEF_RATING",
                    "USG_PCT", "TS_PCT"]]
        .reset_index(drop=True)
    )
    top_net.index += 1
    print("Top 20 by Net Rating, 2024-25")
    display(top_net)
else:
    print("NET_RATING not available")

Top 20 by Net Rating, 2024-25


,PLAYER_NAME,TEAM_ABBREVIATION,GP,MIN,NET_RATING,OFF_RATING,DEF_RATING,USG_PCT,TS_PCT
1,Shai Gilgeous-Alexander,OKC,76,34.200,16.700,122.400,105.700,0.336,0.637
2,Chet Holmgren,OKC,32,27.400,16.100,121.700,105.600,0.214,0.599
3,Isaiah Joe,OKC,74,21.700,15.800,119.900,104.100,0.166,0.617
4,Aaron Wiggins,OKC,76,22.900,14.200,120.700,106.600,0.200,0.596
5,Luguentz Dort,OKC,71,29.200,13.100,120.400,107.300,0.133,0.586
6,Isaiah Hartenstein,OKC,57,27.900,13.000,118.300,105.300,0.165,0.599
7,Kawhi Leonard,LAC,37,31.900,12.400,120.300,107.900,0.280,0.589
8,Evan Mobley,CLE,71,30.500,11.900,120.500,108.600,0.226,0.633
9,Al Horford,BOS,60,27.600,11.500,119.600,108.100,0.136,0.563
10,Dean Wade,CLE,59,21.200,10.700,120.000,109.400,0.101,0.563


---
## 5. Lakers Roster — 2025-26 Season

Pull all players on LAL for the current season. Merge base + advanced to get a full picture of the roster.

In [9]:
# Merge base + advanced on PLAYER_ID, drop duplicate name/team columns
if not stats_2526.empty and not adv_2526.empty:
    adv_cols = [c for c in adv_2526.columns
                if c not in stats_2526.columns or c == "PLAYER_ID"]
    lakers_full = stats_2526.merge(
        adv_2526[adv_cols], on="PLAYER_ID", how="left"
    )
    lakers = (
        lakers_full[lakers_full["TEAM_ABBREVIATION"] == "LAL"]
        .sort_values("MIN", ascending=False)
        .reset_index(drop=True)
    )
    print(f"Lakers roster: {len(lakers)} players")

    display_cols = [c for c in [
        "PLAYER_NAME", "AGE", "GP", "MIN", "PTS", "AST", "REB",
        "FG3_PCT", "TS_PCT", "USG_PCT", "NET_RATING", "PIE"
    ] if c in lakers.columns]
    display(lakers[display_cols])
else:
    # Fall back to base stats only
    lakers = stats_2526[stats_2526["TEAM_ABBREVIATION"] == "LAL"].sort_values("MIN", ascending=False)
    print(f"Lakers (base only): {len(lakers)} players")
    display(lakers)

Lakers roster: 18 players


,PLAYER_NAME,AGE,GP,MIN,PTS,AST,REB,FG3_PCT,TS_PCT,USG_PCT,NET_RATING,PIE
0,Luka Dončić,27.000,64,35.800,33.500,8.300,7.700,0.366,0.616,0.368,3.800,0.188
1,Austin Reaves,27.000,51,34.500,23.300,5.500,4.700,0.360,0.641,0.258,5.000,0.137
2,LeBron James,41.000,60,33.200,20.900,7.200,6.100,0.317,0.594,0.262,2.600,0.154
3,Marcus Smart,32.000,62,28.500,9.300,3.000,2.800,0.331,0.543,0.151,6.800,0.059
4,Rui Hachimura,28.000,68,28.300,11.500,0.800,3.300,0.443,0.620,0.149,2.000,0.073
5,Deandre Ayton,27.000,72,27.200,12.500,0.800,8.000,0.000,0.676,0.164,0.100,0.117
6,Jake LaRavia,24.000,82,25.100,8.200,1.800,4.000,0.321,0.571,0.138,0.000,0.067
7,Luke Kennard,29.000,78,21.500,8.400,2.200,2.300,0.478,0.689,0.131,0.500,0.089
8,Jaxson Hayes,26.000,66,18.300,7.500,0.900,4.100,1.000,0.758,0.124,2.400,0.109
9,Jarred Vanderbilt,27.000,65,17.300,4.400,1.300,4.500,0.293,0.544,0.119,0.200,0.080


---
## 6. Luka's Career Trajectory

Pull advanced stats across all available seasons and trace the progression. Luka's PLAYER_ID from the static registry is 1629029.

In [10]:
LUKA_ID = 1629029

# Seasons available in our config range
TARGET_SEASONS = [s for s in SEASON_STRINGS if int(s[:4]) >= 2018]
print(f"Fetching {len(TARGET_SEASONS)} seasons: {TARGET_SEASONS}")

Fetching 8 seasons: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']


In [11]:
# get_multi_season handles rate limiting and caching per-season
luka_adv_all = N.get_multi_season(N.get_player_advanced, TARGET_SEASONS)

if not luka_adv_all.empty and "PLAYER_ID" in luka_adv_all.columns:
    luka_adv = luka_adv_all[luka_adv_all["PLAYER_ID"] == LUKA_ID].copy()
    print(f"Luka rows across seasons: {len(luka_adv)}")
    traj_cols = [c for c in [
        "SEASON", "TEAM_ABBREVIATION", "AGE", "GP", "MIN",
        "NET_RATING", "OFF_RATING", "DEF_RATING",
        "USG_PCT", "TS_PCT", "AST_PCT", "TO_RATIO", "PIE"
    ] if c in luka_adv.columns]
    display(luka_adv[traj_cols].sort_values("SEASON").reset_index(drop=True))
else:
    print("No multi-season data available")

23:48:03  src.data.nba_stats  INFO  Fetching season 2019-20 ...
23:48:03  src.data.cache  INFO  cache miss -- fetching: nba_player_advanced_2019-20
23:48:03  src.data.cache  INFO  cached 529 rows to nba_player_advanced_2019-20.parquet
23:48:03  src.data.nba_stats  INFO  Fetching season 2020-21 ...
23:48:03  src.data.cache  INFO  cache miss -- fetching: nba_player_advanced_2020-21
23:48:04  src.data.cache  INFO  cached 540 rows to nba_player_advanced_2020-21.parquet
23:48:04  src.data.nba_stats  INFO  Fetching season 2021-22 ...
23:48:04  src.data.cache  INFO  cache miss -- fetching: nba_player_advanced_2021-22
23:48:05  src.data.cache  INFO  cached 605 rows to nba_player_advanced_2021-22.parquet
23:48:05  src.data.nba_stats  INFO  Fetching season 2022-23 ...
23:48:05  src.data.cache  INFO  cache miss -- fetching: nba_player_advanced_2022-23
23:48:07  src.data.cache  INFO  cached 539 rows to nba_player_advanced_2022-23.parquet
23:48:07  src.data.nba_stats  INFO  Fetching season 2023-24 

Luka rows across seasons: 7


,SEASON,TEAM_ABBREVIATION,AGE,GP,MIN,NET_RATING,OFF_RATING,DEF_RATING,USG_PCT,TS_PCT,AST_PCT,PIE
0,2019-20,DAL,21.000,61,33.600,5.300,116.700,111.400,0.355,0.585,0.454,0.194
1,2020-21,DAL,22.000,66,34.300,3.900,116.600,112.700,0.350,0.587,0.425,0.179
2,2021-22,DAL,23.000,65,35.400,3.500,113.800,110.300,0.368,0.571,0.458,0.191
3,2022-23,DAL,24.000,66,36.200,2.100,118.100,116.000,0.368,0.609,0.408,0.202
4,2023-24,DAL,25.000,70,37.500,5.800,119.600,113.800,0.355,0.617,0.428,0.200
5,2024-25,LAL,26.000,50,35.400,9.100,119.600,110.500,0.328,0.587,0.342,0.170
6,2025-26,LAL,27.000,64,35.800,3.800,119.400,115.600,0.368,0.616,0.387,0.188


In [12]:
# Also pull base scoring stats across seasons for the career arc chart
luka_base_all = N.get_multi_season(N.get_player_stats, TARGET_SEASONS)

if not luka_base_all.empty and "PLAYER_ID" in luka_base_all.columns:
    luka_base = (
        luka_base_all[luka_base_all["PLAYER_ID"] == LUKA_ID]
        .sort_values("SEASON")
        .reset_index(drop=True)
    )
    display_cols = [c for c in [
        "SEASON", "TEAM_ABBREVIATION", "AGE", "GP", "MIN",
        "PTS", "AST", "REB", "STL", "BLK", "TOV",
        "FG_PCT", "FG3_PCT", "FT_PCT"
    ] if c in luka_base.columns]
    print("Luka career stats (per game)")
    display(luka_base[display_cols])
else:
    print("No base career data available")

23:48:09  src.data.nba_stats  INFO  Fetching season 2019-20 ...
23:48:09  src.data.cache  INFO  cache miss -- fetching: nba_player_stats_2019-20_PerGame
23:48:10  src.data.cache  INFO  cached 529 rows to nba_player_stats_2019-20_PerGame.parquet
23:48:10  src.data.nba_stats  INFO  Fetching season 2020-21 ...
23:48:10  src.data.cache  INFO  cache miss -- fetching: nba_player_stats_2020-21_PerGame
23:48:11  src.data.cache  INFO  cached 540 rows to nba_player_stats_2020-21_PerGame.parquet
23:48:11  src.data.nba_stats  INFO  Fetching season 2021-22 ...
23:48:11  src.data.cache  INFO  cache miss -- fetching: nba_player_stats_2021-22_PerGame
23:48:12  src.data.cache  INFO  cached 605 rows to nba_player_stats_2021-22_PerGame.parquet
23:48:12  src.data.nba_stats  INFO  Fetching season 2022-23 ...
23:48:12  src.data.cache  INFO  cache miss -- fetching: nba_player_stats_2022-23_PerGame
23:48:13  src.data.cache  INFO  cached 539 rows to nba_player_stats_2022-23_PerGame.parquet
23:48:13  src.data.n

Luka career stats (per game)


,SEASON,TEAM_ABBREVIATION,AGE,GP,MIN,PTS,AST,REB,STL,BLK,TOV,FG_PCT,FG3_PCT,FT_PCT
0,2019-20,DAL,21.000,61,33.600,28.800,8.800,9.400,1.000,0.200,4.300,0.463,0.316,0.758
1,2020-21,DAL,22.000,66,34.300,27.700,8.600,8.000,1.000,0.500,4.300,0.479,0.350,0.730
2,2021-22,DAL,23.000,65,35.400,28.400,8.700,9.100,1.200,0.600,4.500,0.457,0.353,0.744
3,2022-23,DAL,24.000,66,36.200,32.400,8.000,8.600,1.400,0.500,3.600,0.496,0.342,0.742
4,2023-24,DAL,25.000,70,37.500,33.900,9.800,9.200,1.400,0.500,4.000,0.487,0.382,0.786
5,2024-25,LAL,26.000,50,35.400,28.200,7.700,8.200,1.800,0.400,3.600,0.450,0.368,0.782
6,2025-26,LAL,27.000,64,35.800,33.500,8.300,7.700,1.600,0.500,4.000,0.476,0.366,0.780


---
## 7. Shooting Splits — Catch & Shoot vs Pull-Up

These are the columns we'll need for the Luka Complement Score in Phase 1. Catch-and-shoot volume + efficiency is the single most important indicator of a Luka-compatible wing.

In [13]:
shooting_2425 = N.get_player_shooting(SEASON_PREV)
print(f"Shooting rows: {len(shooting_2425)}")
print(f"Columns: {list(shooting_2425.columns)}")
shooting_2425.head(3)

23:48:15  src.data.cache  INFO  cache miss -- fetching: nba_shooting_catchshoot_2024-25
23:48:16  src.data.cache  INFO  cached 569 rows to nba_shooting_catchshoot_2024-25.parquet
23:48:16  src.data.cache  INFO  cache miss -- fetching: nba_shooting_pullup_2024-25
23:48:24  src.data.cache  INFO  cached 569 rows to nba_shooting_pullup_2024-25.parquet


Shooting rows: 569
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'CS_W', 'CS_L', 'CS_MIN', 'CS_CATCH_SHOOT_FGM', 'CS_CATCH_SHOOT_FGA', 'CS_CATCH_SHOOT_FG_PCT', 'CS_CATCH_SHOOT_PTS', 'CS_CATCH_SHOOT_FG3M', 'CS_CATCH_SHOOT_FG3A', 'CS_CATCH_SHOOT_FG3_PCT', 'CS_CATCH_SHOOT_EFG_PCT', 'GP_pu_dup', 'PU_W', 'PU_L', 'PU_MIN', 'PU_PULL_UP_FGM', 'PU_PULL_UP_FGA', 'PU_PULL_UP_FG_PCT', 'PU_PULL_UP_PTS', 'PU_PULL_UP_FG3M', 'PU_PULL_UP_FG3A', 'PU_PULL_UP_FG3_PCT', 'PU_PULL_UP_EFG_PCT']


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,CS_W,CS_L,CS_MIN,CS_CATCH_SHOOT_FGM,CS_CATCH_SHOOT_FGA,CS_CATCH_SHOOT_FG_PCT,CS_CATCH_SHOOT_PTS,CS_CATCH_SHOOT_FG3M,CS_CATCH_SHOOT_FG3A,CS_CATCH_SHOOT_FG3_PCT,CS_CATCH_SHOOT_EFG_PCT,GP_pu_dup,PU_W,PU_L,PU_MIN,PU_PULL_UP_FGM,PU_PULL_UP_FGA,PU_PULL_UP_FG_PCT,PU_PULL_UP_PTS,PU_PULL_UP_FG3M,PU_PULL_UP_FG3A,PU_PULL_UP_FG3_PCT,PU_PULL_UP_EFG_PCT
0,2544,LeBron James,1610612747,LAL,70,44,26,34.900,1.400,3.200,0.429,4.000,1.200,2.900,0.420,0.619,70,44,26,34.900,2.400,6.100,0.386,5.500,0.800,2.500,0.326,0.451
1,101108,Chris Paul,1610612759,SAS,82,34,48,28.000,0.600,1.700,0.380,1.900,0.600,1.600,0.381,0.566,82,34,48,28.000,2.100,5.100,0.424,5.400,1.100,2.900,0.370,0.529
2,200768,Kyle Lowry,1610612755,PHI,35,14,21,18.800,0.600,1.800,0.328,1.800,0.600,1.800,0.333,0.492,35,14,21,18.800,0.300,1.000,0.333,0.900,0.200,0.700,0.320,0.444


In [14]:
# Top catch-and-shoot 3PT shooters by volume x efficiency
cs_3pt_col = next((c for c in shooting_2425.columns if "CS_" in c and "FG3_PCT" in c), None)
cs_3pa_col = next((c for c in shooting_2425.columns if "CS_" in c and "FG3A" in c and "FREQ" not in c), None)

if cs_3pt_col and cs_3pa_col and not shooting_2425.empty:
    cs_shooters = (
        shooting_2425.dropna(subset=[cs_3pt_col, cs_3pa_col])
        .query(f"{cs_3pa_col} >= 1.5")  # min 1.5 CS 3PA per game
        .sort_values(cs_3pt_col, ascending=False)
        .head(20)[["PLAYER_NAME", "TEAM_ABBREVIATION", cs_3pa_col, cs_3pt_col]]
        .reset_index(drop=True)
    )
    cs_shooters.index += 1
    print("Top CS-3PT shooters (min 1.5 CS 3PA/g), 2024-25")
    display(cs_shooters)
else:
    print("Catch-and-shoot columns not available -- check what came back:")
    print([c for c in shooting_2425.columns if "CS" in c or "3PT" in c or "FG3" in c])

Top CS-3PT shooters (min 1.5 CS 3PA/g), 2024-25


,PLAYER_NAME,TEAM_ABBREVIATION,CS_CATCH_SHOOT_FG3A,CS_CATCH_SHOOT_FG3_PCT
1,P.J. Tucker,NYK,1.700,0.600
2,Dru Smith,MIA,1.700,0.583
3,Daeqwon Plowden,ATL,2.800,0.529
4,Dante Exum,DAL,2.100,0.500
5,Dariq Whitehead,BKN,3.200,0.500
6,Taurean Prince,MIL,3.300,0.498
7,Luke Kennard,MEM,3.000,0.469
8,Karl-Anthony Towns,NYK,3.900,0.462
9,Kevin Durant,PHX,4.800,0.456
10,Vít Krejčí,ATL,2.800,0.456


---
## 8. Defensive Stats — Column Discovery

In [15]:
defense_2425 = N.get_player_defense(SEASON_PREV)
print(f"Defense rows: {len(defense_2425)}")
print(f"Columns: {list(defense_2425.columns)}")
defense_2425.head(3)

23:48:24  src.data.cache  INFO  cache miss -- fetching: nba_player_defense_2024-25
23:48:33  src.data.cache  INFO  cached 568 rows to nba_player_defense_2024-25.parquet


Defense rows: 568
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'PLAYER_LAST_TEAM_ID', 'PLAYER_LAST_TEAM_ABBREVIATION', 'PLAYER_POSITION', 'AGE', 'GP', 'G', 'FREQ', 'D_FGM', 'D_FGA', 'D_FG_PCT', 'NORMAL_FG_PCT', 'PCT_PLUSMINUS']


,PLAYER_ID,PLAYER_NAME,PLAYER_LAST_TEAM_ID,PLAYER_LAST_TEAM_ABBREVIATION,PLAYER_POSITION,AGE,GP,G,FREQ,D_FGM,D_FGA,D_FG_PCT,NORMAL_FG_PCT,PCT_PLUSMINUS
0,203999,Nikola Jokić,1610612743,DEN,C,30.000,70,70,1.000,10.030,20.470,0.490,0.494,-0.004
1,1631117,Walker Kessler,1610612762,UTA,C,23.000,58,58,1.000,8.660,19.720,0.439,0.488,-0.049
2,201572,Brook Lopez,1610612749,MIL,C,37.000,80,80,1.000,8.910,19.050,0.468,0.488,-0.020


In [16]:
# Best close defenders: lowest opponent FG% vs league average (PCT_PLUSMINUS most negative)
if not defense_2425.empty and "PCT_PLUSMINUS" in defense_2425.columns:
    best_defenders = (
        defense_2425[defense_2425["GP"] >= 30]
        .sort_values("PCT_PLUSMINUS")
        .head(20)[["PLAYER_NAME", "PLAYER_LAST_TEAM_ABBREVIATION", "GP",
                    "D_FGA", "D_FG_PCT", "NORMAL_FG_PCT", "PCT_PLUSMINUS"]]
        .reset_index(drop=True)
    )
    best_defenders.index += 1
    print("Top defenders by opponent FG% suppression, 2024-25 (negative PCT_PLUSMINUS = better)")
    display(best_defenders)
else:
    print("PCT_PLUSMINUS not available -- columns:", list(defense_2425.columns))

Top defenders by opponent FG% suppression, 2024-25 (negative PCT_PLUSMINUS = better)


,PLAYER_NAME,PLAYER_LAST_TEAM_ABBREVIATION,GP,D_FGA,D_FG_PCT,NORMAL_FG_PCT,PCT_PLUSMINUS
1,Taj Gibson,CHA,36,5.500,0.364,0.479,-0.115
2,Jae'Sean Tate,HOU,47,4.450,0.392,0.468,-0.076
3,Nicolas Batum,LAC,78,6.380,0.396,0.470,-0.075
4,Alex Caruso,OKC,54,8.960,0.395,0.467,-0.072
5,Chet Holmgren,OKC,32,14.810,0.418,0.488,-0.071
6,DeAndre Jordan,DEN,53,5.960,0.421,0.491,-0.070
7,Jalen Pickett,DEN,43,5.490,0.386,0.454,-0.069
8,Victor Wembanyama,SAS,46,18.430,0.427,0.493,-0.066
9,Jaren Jackson Jr.,MEM,73,13.030,0.419,0.480,-0.061
10,Rayan Rupert,POR,39,4.030,0.389,0.450,-0.061


---
## 9. Cache Status

Confirm all data is now cached locally so future notebooks don't hit the API.

In [17]:
info = C.cache_info()
print(f"Cached files: {len(info)}")
display(info)

Cached files: 19


,file,key,size_kb,modified
0,nba_player_advanced_2019-20.parquet,nba_player_advanced_2019-20,201.500,2026-05-27 23:48
1,nba_player_advanced_2020-21.parquet,nba_player_advanced_2020-21,204.500,2026-05-27 23:48
2,nba_player_advanced_2021-22.parquet,nba_player_advanced_2021-22,219.200,2026-05-27 23:48
3,nba_player_advanced_2022-23.parquet,nba_player_advanced_2022-23,203.700,2026-05-27 23:48
4,nba_player_advanced_2023-24.parquet,nba_player_advanced_2023-24,212.400,2026-05-27 23:48
5,nba_player_advanced_2024-25.parquet,nba_player_advanced_2024-25,212.900,2026-05-27 23:48
6,nba_player_advanced_2025-26.parquet,nba_player_advanced_2025-26,216.100,2026-05-27 23:48
7,nba_player_advanced_2026-27.parquet,nba_player_advanced_2026-27,33.500,2026-05-27 23:48
8,nba_player_defense_2024-25.parquet,nba_player_defense_2024-25,31.400,2026-05-27 23:48
9,nba_player_stats_2019-20_PerGame.parquet,nba_player_stats_2019-20_PerGame,139.200,2026-05-27 23:48


---
## 10. Data Quality Gaps — Summary

Key observations for Phase 0.2 planning:

In [18]:
# Confirm what's missing vs what we hoped to have
desired_advanced = ["PER", "WS", "WS_48", "BPM", "VORP", "PIE", "NET_RATING",
                    "OFF_RATING", "DEF_RATING", "TS_PCT", "USG_PCT"]

print("=== Advanced metric availability (nba_api) ===")
for col in desired_advanced:
    available = col in adv_2425.columns if not adv_2425.empty else False
    status = "YES" if available else "NO  (add BBRef in Phase 0.2)"
    print(f"  {col:<15} {status}")

print()
print("=== Shooting splits availability ===")
for prefix, label in [("CS_", "Catch-and-shoot"), ("PU_", "Pull-up")]:
    cols = [c for c in shooting_2425.columns if c.startswith(prefix)]
    print(f"  {label}: {len(cols)} columns -- {cols[:5]}{'...' if len(cols) > 5 else ''}")

print()
print("=== Gaps requiring Phase 0.2 ===")
gaps = [
    "PER, WS/48, BPM, VORP -- Basketball Reference ingestion",
    "EPM, RAPTOR -- Dunks & Threes / FiveThirtyEight archive",
    "Salary data -- Spotrac scraper or manual CSV (load_salaries())",
    "On/off splits -- leaguedashlineups or synergy endpoints",
    "Play-type data -- synergyplaytypes endpoint (drives, post, P&R)",
]
for g in gaps:
    print(f"  - {g}")

=== Advanced metric availability (nba_api) ===
  PER             NO  (add BBRef in Phase 0.2)
  WS              NO  (add BBRef in Phase 0.2)
  WS_48           NO  (add BBRef in Phase 0.2)
  BPM             NO  (add BBRef in Phase 0.2)
  VORP            NO  (add BBRef in Phase 0.2)
  PIE             YES
  NET_RATING      YES
  OFF_RATING      YES
  DEF_RATING      YES
  TS_PCT          YES
  USG_PCT         YES

=== Shooting splits availability ===
  Catch-and-shoot: 11 columns -- ['CS_W', 'CS_L', 'CS_MIN', 'CS_CATCH_SHOOT_FGM', 'CS_CATCH_SHOOT_FGA']...
  Pull-up: 11 columns -- ['PU_W', 'PU_L', 'PU_MIN', 'PU_PULL_UP_FGM', 'PU_PULL_UP_FGA']...

=== Gaps requiring Phase 0.2 ===
  - PER, WS/48, BPM, VORP -- Basketball Reference ingestion
  - EPM, RAPTOR -- Dunks & Threes / FiveThirtyEight archive
  - Salary data -- Spotrac scraper or manual CSV (load_salaries())
  - On/off splits -- leaguedashlineups or synergy endpoints
  - Play-type data -- synergyplaytypes endpoint (drives, post, P&R)
